<a href="https://colab.research.google.com/github/Luseat/automatic_dataset_labeling/blob/main/Labeled_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install kagglehub
import os
import kagglehub
import shutil

os.environ['KAGGLE_API_TOKEN'] = "KGAT_42a729a11399080b96f3c3b8cc516291"

# Download dataset
print("Mulai download dataset...")
path = kagglehub.dataset_download("modassirafzal/anime-faces")

# Pindahin ke folder raw_images biar gampang diakses
shutil.copytree(path, "raw_images", dirs_exist_ok=True)
print("Berhasil! Semua gambar udah ada di folder 'raw_images'")

Mulai download dataset...


100%|██████████| 510M/510M [00:29<00:00, 18.3MB/s]

Extracting files...


Berhasil! Semua gambar udah ada di folder 'raw_images'


Kalau Cell 1 udah kelar 100%, bikin kotak kode baru (+ Code). Bikin 7 folder emosi buat nampung hasil sortirannya

In [4]:
import os

emotions = ['Happy', 'Sad', 'Angry', 'Fear', 'Surprise', 'Disgust', 'Neutral']
os.makedirs('labeled_images', exist_ok=True)

for em in emotions:
  os.makedirs(f'labeled_images/{em}', exist_ok=True)

print("Folder berhasil dibuat...")

Folder berhasil dibuat...


In [6]:
!pip install transformers torch torchvision tqdm # AI CLIP buatan OpenAI

In [ ]:
from transformers import pipeline
from PIL import Image
import os
import shutil
from tqdm import tqdm

print("Mempersiapkan AI CLIP (Makan waktu bentar buat download model)...")

classifier = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32", device=0)

labels = [
    "anime face smilling happy",
    "anime face crying sad",
    "anime face angry mad",
    "anime face scared fear",
    "anime face surprised shoked",
    "anime face disgusted gross",
    "anime face neutral expresionless"
]

label_map = {
    "anime face smilling happy": "Happy",
    "anime face crying sad": "Sad",
    "anime face angry mad": "Angry",
    "anime face scared fear": "Fear",
    "anime face surprised shoked": "surprised",
    "anime face disgusted gross": "disgusted",
    "anime face neutral expresionless": "neutral"
}



source_folder = "labeling_dataset"
images = os.listdir(source_folder)

TARGET_PER_CLASS = 5000
counts = {em: 0 for em in label_map.values()}

print("Mulai menyotir puluhan ribu gambar...")
for img_name in tqdm(images):
  # Kalo semua folder udah dapet 5000 gambar, otomatis berhenti
  if all (v >= TARGET_PER_CLASS for v in counts.values()):
    print("\nTarget 15.000 gambar sudah tercapai...")
    break


  img_path = os.path.join(source_folder, img_name)
  try:
    img = Image.open(img_path).convert('RGB')
    preds = classifier(img, candidate_labels=labels)
    top_pred = preds[0]



    # Kalau AI YAKIN di atas 70% sama tebakannya, baru kita masukin folder
    if top_pred['score'] > 0.70:
      emotion = label_map[top_pred['label']]
      if counts[emotion] < TARGET_PER_CLASS:
        dest_path = os.path.join('labeled_images', emotion, img_name)
        shutil.copy(img_path, dest_path)
        counts[emotion] += 1

  except Exception as e:
    continue

print("Selesai, jumlah per emosi:", counts)
